# InsightFace Face Recognition Colab

這份 notebook 會在 Google Colab 上安裝並執行 [deepinsight/insightface](https://github.com/deepinsight/insightface)，用來比對兩張照片中的臉是否可能是同一人。

流程：

1. 安裝 InsightFace 與 ONNX Runtime GPU。
2. 載入 `buffalo_l` 模型包。
3. 上傳兩張照片。
4. 每張照片取面積最大的臉。
5. 計算兩個 face embedding 的 cosine similarity。

注意：InsightFace 程式碼是 MIT License，但官方 README 說明部分訓練資料與模型僅供非商業研究用途。正式商用前請確認模型授權。

## 1. 設定 Colab Runtime 與安裝套件

在 Colab 選單選：`Runtime` -> `Change runtime type` -> `T4 GPU` 或其他 GPU。

InsightFace 0.7.3 的 Cython 模組是用 NumPy 1.x C-API 編出來的，但 Colab 預裝的 OpenCV / JAX / shap 等套件會強制要求 NumPy 2.x，兩邊衝突會出現 `np._core._multiarray_umath has no attribute _blas_supports_fpe` 之類的錯誤。

下面的安裝 cell 會：

1. 把 NumPy、OpenCV、ONNX Runtime、InsightFace 一次鎖到彼此相容的版本。
2. 安裝完成後**自動重啟 runtime**（這個重啟是必要的，否則舊的 NumPy 還在記憶體裡）。

重啟後**請從第 2 段（匯入套件）的 cell 開始往下執行，不要再執行這個安裝 cell**。

In [ ]:
!pip install -q \
    "numpy==1.26.4" \
    "opencv-python==4.10.0.84" \
    "opencv-python-headless==4.10.0.84" \
    "opencv-contrib-python==4.10.0.84" \
    "onnxruntime-gpu==1.19.2" \
    "onnx==1.16.1" \
    "insightface==0.7.3" \
    "pillow==10.4.0" \
    "scikit-image==0.24.0" \
    "scikit-learn==1.5.2" \
    "scipy==1.13.1" \
    "protobuf==4.25.5" \
    "albumentations==1.4.18" \
    "matplotlib" \
    --force-reinstall --no-cache-dir

import os

print("\n安裝完成。即將自動重啟 Colab runtime；重啟後請從第 2 段（匯入套件）的 cell 開始往下執行。")
os.kill(os.getpid(), 9)

## 2. 匯入套件並載入模型

⚠️ **這一段是 runtime 重啟之後的起點。** 上面的安裝 cell 跑完會把 runtime 殺掉，你會看到 Colab 顯示「Your session crashed」之類的訊息，那是預期行為，不是錯誤。

請從這個 cell 開始往下執行（不要再執行上面的安裝 cell）。

第一次執行 `FaceAnalysis(name="buffalo_l")` 時，InsightFace 會自動下載模型到 `~/.insightface/models/`。

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import onnxruntime as ort

from google.colab import files
from insightface.app import FaceAnalysis

print("NumPy:", np.__version__)
print("OpenCV:", cv2.__version__)
print("ONNX Runtime:", ort.__version__)

if not np.__version__.startswith("1."):
    raise RuntimeError(
        "NumPy 是 2.x，但 InsightFace 0.7.3 需要 NumPy 1.x。\n"
        "請從 Cell 1（安裝）重新跑一次（包含自動重啟），再從這裡繼續。"
    )

available_providers = ort.get_available_providers()
providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if "CUDAExecutionProvider" in available_providers else ["CPUExecutionProvider"]
ctx_id = 0 if "CUDAExecutionProvider" in providers else -1

print("ONNX Runtime providers:", available_providers)
print("Using providers:", providers)

app = FaceAnalysis(name="buffalo_l", providers=providers)
app.prepare(ctx_id=ctx_id, det_size=(640, 640))

## 3. 上傳比對照片（最多 15 張）

執行下方 cell 後，一次選取多張要被比對的照片（建議 1–15 張）。
每張照片若含多張臉，程式會自動選面積最大的臉。

In [ ]:
MAX_COMPARE = 15

uploaded_compare = files.upload()
compare_paths = list(uploaded_compare.keys())

if len(compare_paths) == 0:
    raise ValueError("請至少上傳一張比對照片。")
if len(compare_paths) > MAX_COMPARE:
    print(f"⚠️  上傳了 {len(compare_paths)} 張，超過上限 {MAX_COMPARE} 張，只取前 {MAX_COMPARE} 張。")
    compare_paths = compare_paths[:MAX_COMPARE]

print(f"✅ 已上傳 {len(compare_paths)} 張比對照片：")
for i, p in enumerate(compare_paths, 1):
    print(f"  [{i:02d}] {p}")

## 4. 上傳目標照片（Target）

執行下方 cell 後，選取**一張**目標照片。所有比對照片都會和這張照片做人臉相似度比對。

In [ ]:
uploaded_target = files.upload()
target_paths = list(uploaded_target.keys())

if len(target_paths) != 1:
    raise ValueError(f"請剛好上傳一張目標照片，目前上傳了 {len(target_paths)} 張。")

target_path = target_paths[0]
print(f"✅ 目標照片：{target_path}")

## 5. 定義比對工具函式

In [ ]:
def read_image(path):
    image = cv2.imread(path)
    if image is None:
        raise ValueError(f"讀不到圖片：{path}")
    return image


def face_area(face):
    x1, y1, x2, y2 = face.bbox
    return float((x2 - x1) * (y2 - y1))


def select_largest_face(faces, image_path):
    if len(faces) == 0:
        raise ValueError(f"沒有偵測到臉：{image_path}")
    return max(faces, key=face_area)


def analyze_image(path):
    image = read_image(path)
    faces = app.get(image)
    face = select_largest_face(faces, path)
    return image, face


def cosine_similarity(face_a, face_b):
    return float(np.dot(face_a.normed_embedding, face_b.normed_embedding))


def draw_face_box(image, face, label, color):
    """在圖上畫出選取的人臉框與 label，回傳 RGB numpy array。"""
    output = image.copy()
    x1, y1, x2, y2 = face.bbox.astype(int)
    cv2.rectangle(output, (x1, y1), (x2, y2), color, 3)
    cv2.putText(output, label, (x1, max(y1 - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2)
    if face.kps is not None:
        for kx, ky in face.kps.astype(int):
            cv2.circle(output, (kx, ky), 3, (255, 165, 0), -1)
    return cv2.cvtColor(output, cv2.COLOR_BGR2RGB)

## 6. 執行批次比對，輸出全部結果

`THRESHOLD` 是粗略參考值，不是官方固定標準。光線、角度、遮擋、照片品質都會影響分數；若要用在正式產品，應用自己的資料集校正 threshold。

In [ ]:
THRESHOLD = 0.35   # 可自行調整

# ── 分析目標照片 ──────────────────────────────────────
target_img, target_face = analyze_image(target_path)
print(f"目標照片：{target_path}")
print(f"  偵測到臉 bbox={target_face.bbox.astype(int).tolist()}\n")

# ── 逐張分析並比對 ────────────────────────────────────
results = []   # list of dict
errors  = []

for path in compare_paths:
    try:
        img, face = analyze_image(path)
        sim = cosine_similarity(face, target_face)
        results.append({"path": path, "img": img, "face": face,
                         "sim": sim, "match": sim >= THRESHOLD})
        status = "✅ 同一人" if sim >= THRESHOLD else "❌ 不同人"
        print(f"{status}  sim={sim:.4f}  {path}")
    except Exception as e:
        errors.append(path)
        print(f"⚠️  跳過（{e}）：{path}")

# ── 圖文並排輸出 ──────────────────────────────────────
n = len(results)
COLS = 4
ROWS = (n + COLS - 1) // COLS

fig, axes = plt.subplots(ROWS, COLS, figsize=(COLS * 4, ROWS * 4.5))
axes = np.array(axes).reshape(-1)   # 統一變 1-D

for ax, r in zip(axes, results):
    color = (0, 220, 80) if r["match"] else (220, 50, 50)
    vis = draw_face_box(r["img"], r["face"],
                        f"sim={r['sim']:.3f}", color)
    ax.imshow(vis)
    label = ("✅ 同一人" if r["match"] else "❌ 不同人") + f"\n{r['path']}"
    ax.set_title(label, fontsize=8,
                 color=("green" if r["match"] else "red"))
    ax.axis("off")

for ax in axes[n:]:
    ax.axis("off")

# ── 在最右上角顯示目標照片 ────────────────────────────
target_vis = draw_face_box(target_img, target_face, "TARGET", (30, 144, 255))
fig.add_axes([0.78, 0.92, 0.20, 0.07]).imshow(target_vis)
fig.axes[-1].axis("off")
fig.axes[-1].set_title(f"Target: {target_path}", fontsize=7)

plt.suptitle(
    f"批次比對結果  |  Threshold={THRESHOLD}  |  "
    f"符合 {sum(r['match'] for r in results)}/{n} 張",
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.show()

if errors:
    print(f"\n以下 {len(errors)} 張照片無法處理（沒有偵測到臉或讀取失敗）：")
    for p in errors:
        print(f"  {p}")